In [ ]:
import os, requests, zipfile, gzip, io, shutil

def project_root():
    d = os.getcwd()
    for _ in range(8):
        if os.path.basename(d) in {"182", "183", "184"}:
            for sub in ("data", "DATA"):
                if os.path.isdir(os.path.join(d, sub)):
                    return d
        p = os.path.dirname(d)
        if p == d:
            break
        d = p
    return os.getcwd()

ROOT = project_root()
DATA_SUB = "DATA" if os.path.isdir(os.path.join(ROOT, "DATA")) else "data"
DATA_DIR = os.path.join(ROOT, DATA_SUB, "external")
os.makedirs(DATA_DIR, exist_ok=True)
print("data dir:", DATA_DIR)


In [ ]:
def download(url, name, timeout=180):
    dest = os.path.join(DATA_DIR, name)
    if os.path.exists(dest):
        print("skip", name, os.path.getsize(dest))
        return dest
    r = requests.get(url, stream=True, timeout=timeout)
    r.raise_for_status()
    with open(dest, "wb") as f:
        for chunk in r.iter_content(8192):
            f.write(chunk)
    print("saved", name, os.path.getsize(dest))
    return dest

def _load_kaggle_creds():
    import json as _json
    candidates = []
    if os.environ.get("KAGGLE_CONFIG_DIR"):
        candidates.append(os.environ["KAGGLE_CONFIG_DIR"])
    candidates.append(os.path.join(os.path.expanduser("~"), ".kaggle"))
    candidates.append(os.path.join(ROOT))
    candidates.append(os.path.join(os.path.expanduser("~"), "Downloads"))
    candidates.append(r"C:\Users\ashut\Downloads")
    for c in candidates:
        if not c:
            continue
        for fn in ("kaggle.json", "Kaggle.json"):
            p = os.path.join(c, fn)
            if os.path.exists(p):
                try:
                    d = _json.load(open(p))
                except Exception:
                    continue
                if d.get("key"):
                    os.environ["KAGGLE_API_TOKEN"] = d["key"]
                    os.environ["KAGGLE_KEY"] = d["key"]
                if d.get("username"):
                    os.environ["KAGGLE_USERNAME"] = d["username"]
                return p
    return None

def kaggle_fetch(slug, folder):
    try:
        import kaggle
    except Exception as e:
        print("kaggle package unavailable, skipping", slug, "->", e)
        return None
    _load_kaggle_creds()
    try:
        kaggle.api.authenticate()
    except Exception as e:
        print("kaggle auth failed, skipping", slug, "->", e)
        return None
    dest = os.path.join(DATA_DIR, folder)
    os.makedirs(dest, exist_ok=True)
    try:
        kaggle.api.dataset_download_files(slug, path=dest, unzip=True, quiet=False)
    except Exception as e:
        print("kaggle download failed, skipping", slug, "->", e)
        return None
    print("kaggle fetched", slug, "->", dest)
    return dest


In [ ]:
download("https://snap.stanford.edu/data/soc-sign-bitcoinotc.csv.gz", "bitcoin_otc_trust.csv.gz")
download("https://snap.stanford.edu/data/soc-sign-bitcoinalpha.csv.gz", "bitcoin_alpha_trust.csv.gz")


In [ ]:
kaggle_fetch("ellipticco/elliptic-data-set", "elliptic_aml")


In [ ]:
print("external data ready in", DATA_DIR)
for f in sorted(os.listdir(DATA_DIR)):
    p = os.path.join(DATA_DIR, f)
    print(f, "dir" if os.path.isdir(p) else os.path.getsize(p))
